In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Locate cached full-Z cell files (.npz).
search_roots = [
    Path('/root/capsule/scratch'),
    Path('/root/capsule/code/unmix_filter_qc'),
]
cache_files = []
for root in search_roots:
    if root.exists():
        cache_files.extend(sorted(root.rglob('*_fullz.npz')))

if not cache_files:
    raise FileNotFoundError(
        'No full-Z cache files found. Build cache first, e.g. with build_fullz_cell_crop_cache.py.'
    )

# Pick one example cache file (you can change this selection).
cache_options = {p.name: p for p in cache_files}
file_dropdown = widgets.Dropdown(
    options=list(cache_options.keys()),
    value=list(cache_options.keys())[0],
    description='Cache file:',
    layout=widgets.Layout(width='520px'),
)

z_slider = widgets.IntSlider(description='Z', min=0, max=0, step=1, value=0, continuous_update=False)
overlay_checkbox = widgets.Checkbox(value=True, description='Show target-cell outline')

# Scaling controls
scale_mode = widgets.Dropdown(
    options=['autoscale', 'fixed'],
    value='autoscale',
    description='Scale mode:',
    layout=widgets.Layout(width='220px'),
)
vmin_slider = widgets.FloatSlider(description='vmin pct', min=0, max=20, step=0.5, value=1.0, continuous_update=False)
vmax_slider = widgets.FloatSlider(description='vmax pct', min=80, max=100, step=0.5, value=99.5, continuous_update=False)
fixed_vmin_text = widgets.FloatText(value=100.0, description='fixed vmin', layout=widgets.Layout(width='180px'))
fixed_vmax_text = widgets.FloatText(value=1200.0, description='fixed vmax', layout=widgets.Layout(width='180px'))

status_html = widgets.HTML()
out = widgets.Output()

state = {'images': None, 'channels': None, 'cell_outline': None, 'cell_id': None, 'round_key': None}


def _sync_scale_ui(*_):
    is_auto = (scale_mode.value == 'autoscale')
    vmin_slider.disabled = not is_auto
    vmax_slider.disabled = not is_auto
    fixed_vmin_text.disabled = is_auto
    fixed_vmax_text.disabled = is_auto


def _load_cache_file(path: Path):
    with np.load(path, allow_pickle=False) as d:
        images = d['images']  # (C, Z, Y, X)
        channels = d['channels'].astype(str).tolist()
        cell_outline = d['cell_outline'] if 'cell_outline' in d.files else None
        cell_id = int(d['cell_id']) if 'cell_id' in d.files else None
        round_key = str(d['round_key']) if 'round_key' in d.files else 'unknown'

    if images.ndim != 4:
        raise ValueError(f'Expected images with shape (C,Z,Y,X), got {images.shape}')

    state['images'] = images
    state['channels'] = channels
    state['cell_outline'] = cell_outline
    state['cell_id'] = cell_id
    state['round_key'] = round_key

    z_slider.max = images.shape[1] - 1
    z_slider.value = images.shape[1] // 2
    status_html.value = (
        f"<b>Loaded:</b> {path.name} | "
        f"cell={cell_id} | round={round_key} | "
        f"shape={tuple(images.shape)}"
    )


def _render(*_):
    images = state['images']
    channels = state['channels']
    cell_outline = state['cell_outline']
    if images is None:
        return

    z = int(z_slider.value)
    slices = images[:, z, :, :]

    if scale_mode.value == 'autoscale':
        vmin_pct = float(vmin_slider.value)
        vmax_pct = float(vmax_slider.value)
        if vmin_pct >= vmax_pct:
            with out:
                out.clear_output(wait=True)
                print('vmin pct must be < vmax pct')
            return
        vmins = [np.percentile(s, vmin_pct) for s in slices]
        vmaxs = [np.percentile(s, vmax_pct) for s in slices]
    else:
        fixed_vmin = float(fixed_vmin_text.value)
        fixed_vmax = float(fixed_vmax_text.value)
        if fixed_vmin >= fixed_vmax:
            with out:
                out.clear_output(wait=True)
                print('fixed vmin must be < fixed vmax')
            return
        vmins = [fixed_vmin for _ in slices]
        vmaxs = [fixed_vmax for _ in slices]

    n_ch = len(channels)
    fig, axes = plt.subplots(1, n_ch, figsize=(4 * n_ch, 4.2), constrained_layout=True)
    if n_ch == 1:
        axes = [axes]

    for i, (ax, ch) in enumerate(zip(axes, channels)):
        ax.imshow(slices[i], cmap='gray', vmin=vmins[i], vmax=vmaxs[i])
        if overlay_checkbox.value and cell_outline is not None:
            overlay = cell_outline[z]
            ax.imshow(np.ma.masked_where(overlay == 0, overlay), cmap='autumn', alpha=0.65)
        ax.set_title(f'Ch {ch} | Z={z}')
        ax.axis('off')

    with out:
        out.clear_output(wait=True)
        display(fig)
        plt.close(fig)


def _on_file_change(change):
    if change['name'] == 'value':
        _load_cache_file(cache_options[change['new']])
        _render()


file_dropdown.observe(_on_file_change, names='value')
z_slider.observe(_render, names='value')
overlay_checkbox.observe(_render, names='value')
scale_mode.observe(_sync_scale_ui, names='value')
scale_mode.observe(_render, names='value')
vmin_slider.observe(_render, names='value')
vmax_slider.observe(_render, names='value')
fixed_vmin_text.observe(_render, names='value')
fixed_vmax_text.observe(_render, names='value')

_load_cache_file(cache_options[file_dropdown.value])
_sync_scale_ui()
controls = widgets.VBox([
    file_dropdown,
    widgets.HBox([z_slider, overlay_checkbox]),
    widgets.HBox([scale_mode, vmin_slider, vmax_slider]),
    widgets.HBox([fixed_vmin_text, fixed_vmax_text]),
    status_html,
])
display(controls, out)
_render()

Output()